In [2]:
# ============================================================
# Cell 1 - Imports, seed, device
# ============================================================

import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

from PIL import Image

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
print("Device          :", device)

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))

PyTorch version : 2.10.0+cu128
CUDA available  : True
Device          : cuda
GPU             : Tesla T4


In [4]:
# ============================================================
# Cell 2 - Load the exact same train/val/test splits
# ============================================================

SPLITS_ROOT = (
    "/kaggle/input/datasets/"
    "iwmm10/chestxray-capstone-splits/"
    "capstone_splits/splits"
)

train_df = pd.read_csv(
    os.path.join(SPLITS_ROOT, "train.csv")
)

val_df = pd.read_csv(
    os.path.join(SPLITS_ROOT, "val.csv")
)

test_df = pd.read_csv(
    os.path.join(SPLITS_ROOT, "test.csv")
)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

Train: (18013, 26)
Val  : (3978, 26)
Test : (3904, 26)


In [5]:
# ============================================================
# Cell 3 - Define pathology labels
# ============================================================

LABELS = [
    "Atelectasis",
    "Consolidation",
    "Infiltration",
    "Pneumothorax",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Effusion",
    "Pneumonia",
    "Pleural_Thickening",
    "Cardiomegaly",
    "Nodule",
    "Mass",
    "Hernia"
]

NUM_CLASSES = len(LABELS)

print("Number of classes:", NUM_CLASSES)
print("No_Finding included:", "No_Finding" in LABELS)

Number of classes: 14
No_Finding included: False


In [6]:
# ============================================================
# Cell 4 - NIH image locations
# ============================================================

NIH_DATA_ROOT = (
    "/kaggle/input/datasets/"
    "nih-chest-xrays/data"
)

IMAGE_DIRS = [
    os.path.join(
        NIH_DATA_ROOT,
        f"images_{i:03d}",
        "images"
    )
    for i in range(1, 13)
]

print("Image directories found:")

for directory in IMAGE_DIRS:
    print(
        os.path.basename(
            os.path.dirname(directory)
        ),
        os.path.exists(directory)
    )

Image directories found:
images_001 True
images_002 True
images_003 True
images_004 True
images_005 True
images_006 True
images_007 True
images_008 True
images_009 True
images_010 True
images_011 True
images_012 True


In [7]:
# ============================================================
# Cell 5 - Build image path map
# ============================================================

image_path_map = {}

for directory in IMAGE_DIRS:
    for filename in os.listdir(directory):
        if filename.lower().endswith(".png"):
            image_path_map[filename] = os.path.join(
                directory,
                filename
            )

print("Total indexed images:", len(image_path_map))

Total indexed images: 112120


In [9]:
# ============================================================
# Cell 6 - High-resolution transforms
# ============================================================

IMAGE_SIZE = 320

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomRotation(
        degrees=7
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print("Image size:", IMAGE_SIZE)
print("✓ Train transform ready")
print("✓ Eval transform ready")

Image size: 320
✓ Train transform ready
✓ Eval transform ready


In [10]:
# ============================================================
# Cell 7 - Dataset class
# ============================================================

class ChestXrayDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_path_map,
        labels,
        transform=None
    ):
        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.image_path_map = image_path_map
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        filename = row["Image Index"]

        image_path = self.image_path_map[
            filename
        ]

        image = Image.open(
            image_path
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        targets = row[
            self.labels
        ].values.astype(np.float32)

        targets = torch.tensor(
            targets,
            dtype=torch.float32
        )

        return image, targets


train_dataset = ChestXrayDataset(
    train_df,
    image_path_map,
    LABELS,
    train_transform
)

val_dataset = ChestXrayDataset(
    val_df,
    image_path_map,
    LABELS,
    eval_transform
)

test_dataset = ChestXrayDataset(
    test_df,
    image_path_map,
    LABELS,
    eval_transform
)

print("Train dataset:", len(train_dataset))
print("Val dataset  :", len(val_dataset))
print("Test dataset :", len(test_dataset))

Train dataset: 18013
Val dataset  : 3978
Test dataset : 3904


In [12]:
# ============================================================
# Cell 8 - DataLoaders
# ============================================================

BATCH_SIZE = 16
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Batch size:", BATCH_SIZE)
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

Batch size: 16
Train batches: 1126
Val batches  : 249
Test batches : 244


In [13]:
# ============================================================
# Cell 9 - Build pretrained ConvNeXt-Tiny
# ============================================================

weights = ConvNeXt_Tiny_Weights.DEFAULT

model = convnext_tiny(
    weights=weights
)

# Original ConvNeXt-Tiny classifier ends with a Linear layer.
in_features = model.classifier[2].in_features

# Replace ImageNet's 1000-class classifier
# with our 14-pathology classifier.
model.classifier[2] = nn.Linear(
    in_features,
    NUM_CLASSES
)

model = model.to(device)

print("=" * 60)
print("CONVNEXT-TINY")
print("=" * 60)

print(f"Classifier input features : {in_features}")
print(f"Output classes            : {NUM_CLASSES}")
print(f"Device                    : {next(model.parameters()).device}")
print("\nNew classifier:")
print(model.classifier)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 196MB/s]  


CONVNEXT-TINY
Classifier input features : 768
Output classes            : 14
Device                    : cuda:0

New classifier:
Sequential(
  (0): LayerNorm2d((768,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=768, out_features=14, bias=True)
)


In [ ]:
# ============================================================
# Cell 10 - Parameter count
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("MODEL PARAMETERS")
print("=" * 60)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

In [ ]:
# ============================================================
# Cell 11 - 320x320 forward-pass / memory test
# ============================================================

torch.cuda.empty_cache()

batch_images, batch_targets = next(
    iter(train_loader)
)

print("Before GPU transfer:")
print("Images :", batch_images.shape)
print("Targets:", batch_targets.shape)

batch_images = batch_images.to(
    device,
    non_blocking=True
)

batch_targets = batch_targets.to(
    device,
    non_blocking=True
)

model.eval()

with torch.no_grad():
    with torch.amp.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        logits = model(batch_images)

print("\n" + "=" * 60)
print("FORWARD PASS")
print("=" * 60)

print("Input :", batch_images.shape)
print("Target:", batch_targets.shape)
print("Output:", logits.shape)

assert logits.shape == batch_targets.shape

print("\n✓ ConvNeXt-Tiny forward pass successful")
print("✓ Batch size 16 fits at 320x320")

print(
    f"GPU memory allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    f"GPU memory reserved : "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

In [ ]:
# ============================================================
# Cell 12 - One training-step memory test
# ============================================================

criterion_test = nn.BCEWithLogitsLoss()

optimizer_test = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scaler_test = torch.amp.GradScaler("cuda")

model.train()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

optimizer_test.zero_grad(set_to_none=True)

with torch.amp.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model(batch_images)
    loss = criterion_test(
        logits,
        batch_targets
    )

scaler_test.scale(loss).backward()
scaler_test.step(optimizer_test)
scaler_test.update()

peak_memory = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("=" * 60)
print("TRAINING STEP TEST")
print("=" * 60)

print(f"Loss            : {loss.item():.4f}")
print(f"Peak GPU memory : {peak_memory:.2f} GB")

print("\n✓ Forward pass successful")
print("✓ Backward pass successful")
print("✓ Optimizer step successful")

In [ ]:
# ============================================================
# Cell 13 - Clean ConvNeXt initialization for D3
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

weights = ConvNeXt_Tiny_Weights.DEFAULT

model = convnext_tiny(
    weights=weights
)

in_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(
    in_features,
    NUM_CLASSES
)

model = model.to(device)

print("✓ Fresh ConvNeXt-Tiny initialized")
print("✓ ImageNet pretrained weights")
print("✓ Output classes:", NUM_CLASSES)

In [ ]:
# ============================================================
# Cell 14 - D3 configuration
# ============================================================

EXPERIMENT_NAME = "D3_ConvNeXtTiny_320_BCE"

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.3,
    patience=1,
    min_lr=1e-6
)

scaler = torch.amp.GradScaler("cuda")

CHECKPOINT_PATH = (
    "/kaggle/working/"
    "convnext_tiny_320_best.pth"
)

print("=" * 60)
print("D3 EXPERIMENT CONFIGURATION")
print("=" * 60)

print(f"Experiment      : {EXPERIMENT_NAME}")
print("Architecture    : ConvNeXt-Tiny")
print(f"Resolution      : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch size      : {BATCH_SIZE}")
print("Loss            : BCEWithLogitsLoss")
print("Optimizer       : AdamW")
print(f"Initial LR      : {LEARNING_RATE}")
print(f"Weight decay    : {WEIGHT_DECAY}")
print("Scheduler       : ReduceLROnPlateau")
print(f"Max epochs      : {MAX_EPOCHS}")
print(f"Early stopping  : {EARLY_STOPPING_PATIENCE}")

In [14]:
# ============================================================
# Cell 15 - Evaluation function
# ============================================================

def evaluate_model(model, data_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for images, targets in data_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            targets = targets.to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(images)
                loss = criterion(logits, targets)

            probs = torch.sigmoid(logits)

            running_loss += (
                loss.item() * images.size(0)
            )

            all_targets.append(
                targets.cpu().numpy()
            )

            all_probs.append(
                probs.cpu().numpy()
            )

    avg_loss = (
        running_loss / len(data_loader.dataset)
    )

    all_targets = np.concatenate(
        all_targets,
        axis=0
    )

    all_probs = np.concatenate(
        all_probs,
        axis=0
    )

    per_class_auc = {}

    for i, label in enumerate(LABELS):

        try:
            auc = roc_auc_score(
                all_targets[:, i],
                all_probs[:, i]
            )

        except ValueError:
            auc = np.nan

        per_class_auc[label] = auc

    macro_auc = np.nanmean(
        list(per_class_auc.values())
    )

    return {
        "loss": avg_loss,
        "macro_auc": macro_auc,
        "per_class_auc": per_class_auc,
        "targets": all_targets,
        "probabilities": all_probs
    }


print("✓ Evaluation function ready")

✓ Evaluation function ready


In [ ]:
# ============================================================
# Cell 16 - One training epoch
# ============================================================

def train_one_epoch(
    model,
    data_loader,
    criterion,
    optimizer,
    device,
    scaler
):
    model.train()

    running_loss = 0.0

    for images, targets in data_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        targets = targets.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(images)
            loss = criterion(
                logits,
                targets
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += (
            loss.item() * images.size(0)
        )

    avg_loss = (
        running_loss / len(data_loader.dataset)
    )

    return avg_loss


print("✓ Training function ready")

In [ ]:
# ============================================================
# Cell 17 - D3 ConvNeXt-Tiny training loop
# ============================================================

d3_history = []

best_val_auc = -np.inf
epochs_without_improvement = 0

print("=" * 72)
print(f"STARTING EXPERIMENT: {EXPERIMENT_NAME}")
print("=" * 72)

for epoch in range(1, MAX_EPOCHS + 1):

    current_lr = optimizer.param_groups[0]["lr"]

    print(f"\nEpoch {epoch}/{MAX_EPOCHS}")
    print("-" * 72)
    print(f"Learning Rate   : {current_lr:.6f}")

    # -------------------------
    # Train
    # -------------------------
    train_loss = train_one_epoch(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        scaler=scaler
    )

    # -------------------------
    # Validate
    # -------------------------
    val_results = evaluate_model(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        device=device
    )

    val_loss = val_results["loss"]
    val_auc = val_results["macro_auc"]

    d3_history.append({
        "epoch": epoch,
        "learning_rate": current_lr,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_auc": val_auc
    })

    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Val Loss        : {val_loss:.4f}")
    print(f"Val Macro AUROC : {val_auc:.4f}")

    # -------------------------
    # Save best checkpoint
    # -------------------------
    if val_auc > best_val_auc:

        best_val_auc = val_auc
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_macro_auc": val_auc,
                "labels": LABELS,
                "experiment": EXPERIMENT_NAME,
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE
            },
            CHECKPOINT_PATH
        )

        print(
            f"✓ New best model saved "
            f"(Macro AUROC: {val_auc:.4f})"
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement "
            f"({epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE})"
        )

    # -------------------------
    # Scheduler step
    # -------------------------
    scheduler.step(val_auc)

    new_lr = optimizer.param_groups[0]["lr"]

    if new_lr < current_lr:
        print(
            f"↓ Learning rate reduced: "
            f"{current_lr:.6f} → {new_lr:.6f}"
        )

    # -------------------------
    # Early stopping
    # -------------------------
    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print("\nEarly stopping triggered.")
        break


print("\n" + "=" * 72)
print("D3 TRAINING COMPLETE")
print("=" * 72)

print(f"Best Validation Macro AUROC: {best_val_auc:.4f}")
print(f"Best checkpoint: {CHECKPOINT_PATH}")

In [ ]:
# ============================================================
# Cell 18 - Restore best ConvNeXt checkpoint
# ============================================================

checkpoint_d3 = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint_d3["model_state_dict"]
)

model = model.to(device)
model.eval()

print("Best epoch      :", checkpoint_d3["epoch"])
print("Validation AUROC:", checkpoint_d3["val_macro_auc"])

In [ ]:
# ============================================================
# Cell 19 - Best validation per-class AUROC
# ============================================================

best_d3_val = evaluate_model(
    model,
    val_loader,
    criterion,
    device
)

d3_per_class = pd.DataFrame({
    "Pathology": LABELS,
    "ConvNeXt Val AUROC": [
        best_d3_val["per_class_auc"][label]
        for label in LABELS
    ]
})

d3_per_class = d3_per_class.sort_values(
    "ConvNeXt Val AUROC",
    ascending=False
).reset_index(drop=True)

display(
    d3_per_class.style.format({
        "ConvNeXt Val AUROC": "{:.4f}"
    })
)

## D4 — ConvNeXt-Tiny 320×320 + Focal Loss

Focal Loss was evaluated as an alternative to BCE.

- Best validation Macro AUROC: **0.8145**
- Best epoch: **4**
- BCE validation Macro AUROC: **0.8121**
- Improvement: **+0.0024**

Although Focal Loss produced a slightly higher macro AUROC, it reduced
AUROC for 8 of 14 pathologies. Therefore, BCE was retained as the final
model because it provided more balanced per-class performance.

In [ ]:
# ============================================================
# Cell 25 - Restore best D3 BCE checkpoint
# ============================================================

D3_CHECKPOINT_PATH = (
    "/kaggle/working/"
    "convnext_tiny_320_best.pth"
)

checkpoint_d3 = torch.load(
    D3_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint_d3["model_state_dict"]
)

model = model.to(device)
model.eval()

criterion = nn.BCEWithLogitsLoss()

print("=" * 60)
print("D3 BCE MODEL RESTORED")
print("=" * 60)
print(f"Best epoch      : {checkpoint_d3['epoch']}")
print(f"Validation AUROC: {checkpoint_d3['val_macro_auc']:.4f}")

In [ ]:
# ============================================================
# Cell 26 - D3 validation predictions
# ============================================================

d3_val_results = evaluate_model(
    model,
    val_loader,
    criterion,
    device
)

val_targets = d3_val_results["targets"]
val_probs = d3_val_results["probabilities"]

print("Targets shape      :", val_targets.shape)
print("Probabilities shape:", val_probs.shape)
print(
    f"Validation Macro AUROC: "
    f"{d3_val_results['macro_auc']:.4f}"
)

In [ ]:
# ============================================================
# Cell 27 - D3 fixed threshold baseline
# ============================================================

val_preds_05 = (
    val_probs >= 0.50
).astype(int)

macro_f1_05 = f1_score(
    val_targets,
    val_preds_05,
    average="macro",
    zero_division=0
)

macro_precision_05 = precision_score(
    val_targets,
    val_preds_05,
    average="macro",
    zero_division=0
)

macro_recall_05 = recall_score(
    val_targets,
    val_preds_05,
    average="macro",
    zero_division=0
)

print("=" * 60)
print("D3 VALIDATION - FIXED THRESHOLD 0.50")
print("=" * 60)
print(f"Macro F1        : {macro_f1_05:.4f}")
print(f"Macro Precision : {macro_precision_05:.4f}")
print(f"Macro Recall    : {macro_recall_05:.4f}")

In [ ]:
# ============================================================
# Cell 28 - D3 threshold optimization
# Validation set ONLY
# ============================================================

threshold_grid = np.arange(
    0.01,
    0.96,
    0.01
)

optimized_thresholds = {}
threshold_results = []

for i, label in enumerate(LABELS):

    y_true = val_targets[:, i]
    y_prob = val_probs[:, i]

    best_threshold = 0.50
    best_f1 = -1.0
    best_precision = 0.0
    best_recall = 0.0

    for threshold in threshold_grid:

        y_pred = (
            y_prob >= threshold
        ).astype(int)

        current_f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

        if current_f1 > best_f1:

            best_f1 = current_f1
            best_threshold = threshold

            best_precision = precision_score(
                y_true,
                y_pred,
                zero_division=0
            )

            best_recall = recall_score(
                y_true,
                y_pred,
                zero_division=0
            )

    optimized_thresholds[label] = float(
        best_threshold
    )

    threshold_results.append({
        "Pathology": label,
        "Threshold": best_threshold,
        "F1": best_f1,
        "Precision": best_precision,
        "Recall": best_recall
    })


threshold_df = pd.DataFrame(
    threshold_results
)

display(
    threshold_df.style.format({
        "Threshold": "{:.2f}",
        "F1": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}"
    })
)

In [ ]:
# ============================================================
# Cell 29 - D3 validation threshold comparison
# ============================================================

threshold_array = np.array([
    optimized_thresholds[label]
    for label in LABELS
])

val_preds_opt = (
    val_probs >= threshold_array
).astype(int)

macro_f1_opt = f1_score(
    val_targets,
    val_preds_opt,
    average="macro",
    zero_division=0
)

macro_precision_opt = precision_score(
    val_targets,
    val_preds_opt,
    average="macro",
    zero_division=0
)

macro_recall_opt = recall_score(
    val_targets,
    val_preds_opt,
    average="macro",
    zero_division=0
)

print("=" * 60)
print("D3 THRESHOLD OPTIMIZATION RESULTS")
print("=" * 60)

print("\nFixed threshold = 0.50")
print(f"Macro F1        : {macro_f1_05:.4f}")
print(f"Macro Precision : {macro_precision_05:.4f}")
print(f"Macro Recall    : {macro_recall_05:.4f}")

print("\nOptimized per-class thresholds")
print(f"Macro F1        : {macro_f1_opt:.4f}")
print(f"Macro Precision : {macro_precision_opt:.4f}")
print(f"Macro Recall    : {macro_recall_opt:.4f}")

In [ ]:
# ============================================================
# Cell 30 - Save D3 optimized thresholds
# ============================================================

import json

D3_THRESHOLD_PATH = (
    "/kaggle/working/"
    "convnext_tiny_320_bce_thresholds.json"
)

with open(D3_THRESHOLD_PATH, "w") as f:
    json.dump(
        optimized_thresholds,
        f,
        indent=2
    )

print("✓ Thresholds frozen and saved")
print(D3_THRESHOLD_PATH)

In [ ]:
# ============================================================
# Cell 31 - FINAL D3 test evaluation
# ============================================================

test_results = evaluate_model(
    model,
    test_loader,
    criterion,
    device
)

test_targets = test_results["targets"]
test_probs = test_results["probabilities"]

test_preds_05 = (
    test_probs >= 0.50
).astype(int)

test_preds_opt = (
    test_probs >= threshold_array
).astype(int)

test_f1_05 = f1_score(
    test_targets,
    test_preds_05,
    average="macro",
    zero_division=0
)

test_precision_05 = precision_score(
    test_targets,
    test_preds_05,
    average="macro",
    zero_division=0
)

test_recall_05 = recall_score(
    test_targets,
    test_preds_05,
    average="macro",
    zero_division=0
)

test_f1_opt = f1_score(
    test_targets,
    test_preds_opt,
    average="macro",
    zero_division=0
)

test_precision_opt = precision_score(
    test_targets,
    test_preds_opt,
    average="macro",
    zero_division=0
)

test_recall_opt = recall_score(
    test_targets,
    test_preds_opt,
    average="macro",
    zero_division=0
)

print("=" * 60)
print("FINAL D3 TEST RESULTS")
print("=" * 60)

print(f"Test Macro AUROC : {test_results['macro_auc']:.4f}")

print("\nFixed threshold = 0.50")
print(f"Macro F1        : {test_f1_05:.4f}")
print(f"Macro Precision : {test_precision_05:.4f}")
print(f"Macro Recall    : {test_recall_05:.4f}")

print("\nValidation-optimized thresholds")
print(f"Macro F1        : {test_f1_opt:.4f}")
print(f"Macro Precision : {test_precision_opt:.4f}")
print(f"Macro Recall    : {test_recall_opt:.4f}")

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [15]:
# ============================================================
# Restore committed D3 checkpoint
# ============================================================

D3_CHECKPOINT_PATH = (
    "/kaggle/input/notebooks/"
    "rabehalmutire/"
    "01-convnext-training/"
    "convnext_tiny_320_best.pth"
)

checkpoint_d3 = torch.load(
    D3_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint_d3["model_state_dict"]
)

model = model.to(device)
model.eval()

criterion = nn.BCEWithLogitsLoss()

print("=" * 60)
print("D3 CHECKPOINT RESTORED")
print("=" * 60)
print(f"Best epoch      : {checkpoint_d3['epoch']}")
print(f"Validation AUROC: {checkpoint_d3['val_macro_auc']:.4f}")

D3 CHECKPOINT RESTORED
Best epoch      : 3
Validation AUROC: 0.8121


In [16]:
# ============================================================
# Load frozen D3 validation thresholds
# ============================================================

import json
import os

THRESHOLD_PATH = (
    "/kaggle/input/notebooks/"
    "rabehalmutire/"
    "01-convnext-training/"
    "convnext_tiny_320_bce_thresholds.json"
)

with open(THRESHOLD_PATH, "r") as f:
    optimized_thresholds = json.load(f)

threshold_array = np.array([
    optimized_thresholds[label]
    for label in LABELS
])

print("✓ Thresholds loaded")
print(optimized_thresholds)

✓ Thresholds loaded
{'Atelectasis': 0.34, 'Consolidation': 0.12, 'Infiltration': 0.17, 'Pneumothorax': 0.29000000000000004, 'Edema': 0.18000000000000002, 'Emphysema': 0.3, 'Fibrosis': 0.18000000000000002, 'Effusion': 0.22, 'Pneumonia': 0.05, 'Pleural_Thickening': 0.23, 'Cardiomegaly': 0.67, 'Nodule': 0.2, 'Mass': 0.19, 'Hernia': 0.5}


In [17]:
# ============================================================
# Final D3 test predictions
# ============================================================

test_results = evaluate_model(
    model,
    test_loader,
    criterion,
    device
)

test_targets = test_results["targets"]
test_probs = test_results["probabilities"]

test_preds = (
    test_probs >= threshold_array
).astype(int)

print(f"Test Macro AUROC: {test_results['macro_auc']:.4f}")

Test Macro AUROC: 0.8158


In [18]:
# ============================================================
# Per-pathology final test metrics
# ============================================================

final_rows = []

for i, label in enumerate(LABELS):

    y_true = test_targets[:, i]
    y_prob = test_probs[:, i]
    y_pred = test_preds[:, i]

    auc = roc_auc_score(
        y_true,
        y_prob
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    final_rows.append({
        "Pathology": label,
        "Threshold": threshold_array[i],
        "AUROC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Test Positives": int(y_true.sum())
    })


final_test_df = pd.DataFrame(final_rows)

final_test_df = final_test_df.sort_values(
    "AUROC",
    ascending=False
).reset_index(drop=True)

display(
    final_test_df.style.format({
        "Threshold": "{:.2f}",
        "AUROC": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}"
    })
)

,Pathology,Threshold,AUROC,Precision,Recall,F1,Test Positives
0,Hernia,0.50,0.9232,0.8182,0.3000,0.4390,30
1,Edema,0.18,0.9207,0.4780,0.6645,0.5560,310
2,Emphysema,0.30,0.9156,0.7204,0.6237,0.6685,380
3,Cardiomegaly,0.67,0.9039,0.5714,0.4861,0.5253,288
4,Pneumothorax,0.29,0.8460,0.3788,0.5707,0.4553,375
5,Effusion,0.22,0.8423,0.4819,0.6922,0.5682,770
6,Mass,0.19,0.8269,0.3609,0.4439,0.3981,374
7,Fibrosis,0.18,0.8101,0.2094,0.3585,0.2643,212
8,Atelectasis,0.34,0.7733,0.3872,0.5556,0.4564,612
9,Nodule,0.20,0.7580,0.3262,0.4715,0.3856,386


In [19]:
FINAL_RESULTS_PATH = (
    "/kaggle/working/"
    "convnext_d3_final_test_metrics.csv"
)

final_test_df.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)

print("✓ Saved:", FINAL_RESULTS_PATH)

✓ Saved: /kaggle/working/convnext_d3_final_test_metrics.csv


In [20]:
# ============================================================
# Cell 32 - Collect logits for calibration
# ============================================================

def collect_logits(model, data_loader, device):
    model.eval()

    all_logits = []
    all_targets = []

    with torch.no_grad():
        for images, targets in data_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(images)

            all_logits.append(
                logits.float().cpu()
            )

            all_targets.append(
                targets.float().cpu()
            )

    return (
        torch.cat(all_logits, dim=0),
        torch.cat(all_targets, dim=0)
    )


val_logits, val_targets_t = collect_logits(
    model,
    val_loader,
    device
)

test_logits, test_targets_t = collect_logits(
    model,
    test_loader,
    device
)

print("Validation logits:", val_logits.shape)
print("Test logits      :", test_logits.shape)

Validation logits: torch.Size([3978, 14])
Test logits      : torch.Size([3904, 14])


In [23]:
# ============================================================
# Cell 33 - Expected Calibration Error
# ============================================================

def binary_ece(
    y_true,
    y_prob,
    n_bins=10
):
    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for lower, upper in zip(
        bin_edges[:-1],
        bin_edges[1:]
    ):

        mask = (
            (y_prob > lower)
            & (y_prob <= upper)
        )

        if mask.sum() == 0:
            continue

        avg_confidence = y_prob[mask].mean()
        actual_rate = y_true[mask].mean()

        weight = mask.mean()

        ece += (
            weight
            * abs(
                avg_confidence
                - actual_rate
            )
        )

    return ece


def multilabel_ece(
    targets,
    probabilities,
    labels
):
    results = {}

    for i, label in enumerate(labels):

        results[label] = binary_ece(
            targets[:, i],
            probabilities[:, i]
        )

    macro_ece = np.mean(
        list(results.values())
    )

    return macro_ece, results

In [25]:
val_probs_uncal = torch.sigmoid(
    val_logits
).numpy()

test_probs_uncal = torch.sigmoid(
    test_logits
).numpy()

val_ece, val_ece_classes = multilabel_ece(
    val_targets_t.numpy(),
    val_probs_uncal,
    LABELS
)

test_ece, test_ece_classes = multilabel_ece(
    test_targets_t.numpy(),
    test_probs_uncal,
    LABELS
)

print("=" * 55)
print("UNCALIBRATED MODEL")
print("=" * 55)

print(f"Validation Macro ECE: {val_ece:.4f}")
print(f"Test Macro ECE      : {test_ece:.4f}")

UNCALIBRATED MODEL
Validation Macro ECE: 0.0245
Test Macro ECE      : 0.0258


In [26]:
# ============================================================
# Cell 35 - Prepare test metadata
# ============================================================

test_meta = test_df.reset_index(drop=True).copy()

assert len(test_meta) == len(test_probs)

test_meta["Patient Gender"].value_counts(dropna=False)

Patient Gender
M    2088
F    1816
Name: count, dtype: int64

In [27]:
# ============================================================
# Cell 36 - Robustness by sex
# ============================================================

sex_results = []

for sex in sorted(test_meta["Patient Gender"].dropna().unique()):

    mask = (
        test_meta["Patient Gender"].values == sex
    )

    subgroup_targets = test_targets[mask]
    subgroup_probs = test_probs[mask]

    per_class_auc = []

    for i, label in enumerate(LABELS):

        y_true = subgroup_targets[:, i]
        y_prob = subgroup_probs[:, i]

        # AUROC requires both positive and negative examples
        if len(np.unique(y_true)) < 2:
            continue

        per_class_auc.append(
            roc_auc_score(y_true, y_prob)
        )

    macro_auc = np.mean(per_class_auc)

    sex_results.append({
        "Sex": sex,
        "Images": int(mask.sum()),
        "Macro AUROC": macro_auc,
        "Classes evaluated": len(per_class_auc)
    })


sex_df = pd.DataFrame(sex_results)

display(
    sex_df.style.format({
        "Macro AUROC": "{:.4f}"
    })
)

,Sex,Images,Macro AUROC,Classes evaluated
0,F,1816,0.8165,14
1,M,2088,0.8138,14


In [28]:
# ============================================================
# Cell 37 - Clean age and define age groups
# ============================================================

test_meta["Patient Age"] = pd.to_numeric(
    test_meta["Patient Age"],
    errors="coerce"
)

# NIH metadata can contain unrealistic ages,
# so keep only a reasonable analysis range.
valid_age_mask = (
    test_meta["Patient Age"].between(0, 120)
)

print(
    "Valid ages:",
    int(valid_age_mask.sum()),
    "/",
    len(test_meta)
)

age_bins = [0, 18, 40, 60, 80, 121]

age_labels = [
    "0-17",
    "18-39",
    "40-59",
    "60-79",
    "80+"
]

test_meta["Age Group"] = pd.cut(
    test_meta["Patient Age"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

print(
    test_meta.loc[
        valid_age_mask,
        "Age Group"
    ].value_counts().sort_index()
)

Valid ages: 3904 / 3904
Age Group
0-17      174
18-39     916
40-59    1772
60-79     988
80+        54
Name: count, dtype: int64


In [29]:
# ============================================================
# Cell 38 - Macro AUROC by age group
# ============================================================

age_results = []

for age_group in age_labels:

    mask = (
        valid_age_mask.values
        & (
            test_meta["Age Group"]
            .astype(str)
            .values
            == age_group
        )
    )

    subgroup_targets = test_targets[mask]
    subgroup_probs = test_probs[mask]

    per_class_auc = []

    for i, label in enumerate(LABELS):

        y_true = subgroup_targets[:, i]
        y_prob = subgroup_probs[:, i]

        if len(np.unique(y_true)) < 2:
            continue

        per_class_auc.append(
            roc_auc_score(
                y_true,
                y_prob
            )
        )

    if len(per_class_auc) == 0:
        continue

    age_results.append({
        "Age Group": age_group,
        "Images": int(mask.sum()),
        "Macro AUROC": np.mean(per_class_auc),
        "Classes evaluated": len(per_class_auc)
    })


age_df = pd.DataFrame(age_results)

display(
    age_df.style.format({
        "Macro AUROC": "{:.4f}"
    })
)

,Age Group,Images,Macro AUROC,Classes evaluated
0,0-17,174,0.8110,13
1,18-39,916,0.8249,14
2,40-59,1772,0.8164,14
3,60-79,988,0.7945,14
4,80+,54,0.7235,13


In [30]:
# ============================================================
# Cell 39 - View-position counts
# ============================================================

print(
    test_meta["View Position"]
    .value_counts(dropna=False)
)

View Position
PA    2263
AP    1641
Name: count, dtype: int64


In [31]:
# ============================================================
# Cell 40 - Macro AUROC by view position
# ============================================================

view_results = []

for view in sorted(
    test_meta["View Position"]
    .dropna()
    .unique()
):

    mask = (
        test_meta["View Position"].values
        == view
    )

    subgroup_targets = test_targets[mask]
    subgroup_probs = test_probs[mask]

    per_class_auc = []

    for i, label in enumerate(LABELS):

        y_true = subgroup_targets[:, i]
        y_prob = subgroup_probs[:, i]

        if len(np.unique(y_true)) < 2:
            continue

        per_class_auc.append(
            roc_auc_score(
                y_true,
                y_prob
            )
        )

    view_results.append({
        "View Position": view,
        "Images": int(mask.sum()),
        "Macro AUROC": np.mean(per_class_auc),
        "Classes evaluated": len(per_class_auc)
    })


view_df = pd.DataFrame(view_results)

display(
    view_df.style.format({
        "Macro AUROC": "{:.4f}"
    })
)

,View Position,Images,Macro AUROC,Classes evaluated
0,AP,1641,0.7989,14
1,PA,2263,0.8084,14
